# Breakdown Harness Engineering in Claude Code

## Initialization and Configuration

In [ ]:
import os
import subprocess

from dataclasses import dataclass, field
from pathlib import Path

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv

In [ ]:
"""
Get Environment Variable
1. ANTHROPIC_API_KEY
2. ANTHROPIC_BASE_URL
3. ANTHROPIC_MODEL_ID
"""

# override覆蓋.env中的同名變數
load_dotenv(override=True)

WORKDIR = Path.cwd()
# getenv 不一定要有； environ 是一定要有否則無法執行
client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
)
MODEL = os.environ["ANTHROPIC_MODEL_ID"]
MAX_TOKEN = 8000
PLAN_REMINDER_INTERVAL = 3

In [ ]:
# System Prompt
SYSTEM = f"""You are a coding agent at {WORKDIR}.\n\nUse the todo tool for multi-step work.\nWrite a short plan before acting when the task has multiple steps.\nKeep exactly one step in_progress at a time.\nRefresh the plan whenever progress changes.\nPrefer tools over prose."""


## S03 Tasks

**痛點**
多步任务容易走一步忘一步
明明已经做过的检查，会重复再做

一口气列出很多步骤后，很快又回到即兴发挥
这是因为模型虽然“能想”，但它的当前注意力始终受上下文影响。
如果没有一块显式、稳定、可反复更新的计划状态，大任务就很容易漂。
所以这一章要补上的，不是“更强的工具”，而是：
**让 agent 把当前会话里的计划外显出来，并且持续更新。**

**名詞解釋**
- 为了完成当前这次请求，先把接下来几步写出来，并在过程中不断更新。
- TODO: 模型用來寫入當前計劃的一條入口

**流程**

```text
用户提出大任务
   |
   v
模型先写一份当前计划
   |
   v
计划状态
  - [ ] 还没做
  - [>] 正在做
  - [x] 已完成
   |
   v
每做完一步，就更新计划
```


In [ ]:
@dataclass
class PlanningState:
    items: list[dict] = field(default_factory=list)
    rounds_since_update: int = 0

In [ ]:
# 任務管理器
class TodoManager:
    def __init__(self):
        self.state = PlanningState()

    def update(self, items: list) -> str:
        """
        更新目前的待辦事項
        """
        if len(items) > 12:
            raise ValueError("Keep the session plan short (max 12 items)")

        normalized = []
        in_progress_count = 0

        for index, raw_item in enumerate(items):
            content = str(raw_item.get("content", "")).strip()
            status = str(raw_item.get("status", "pending")).strip()
            active_form = str(raw_item.get("activeForm", "")).strip()

            if not content:
                raise ValueError(f"Item {index}: content required")
            if status not in {"pending", "in_progress", "completed"}:
                raise ValueError(f"Item {index}: invalid status '{status}'")
            if status == "in_progress":
                in_progress_count += 1

            normalized.append({
                "content": content,
                "status": status,
                "activeForm": active_form,
            })

        if in_progress_count > 1:
            raise ValueError("Only one item can be in_progress")

        self.state.items = normalized
        self.mark_updated()
        return self.render()

    def mark_updated(self) -> None:
        self.state.rounds_since_update = 0

    def note_round_without_update(self) -> None:
        """
        新增Rounds
        """
        self.state.rounds_since_update += 1

    def reminder(self) -> str | None:
        if not self.state.items:
            return None
        if self.state.rounds_since_update < PLAN_REMINDER_INTERVAL:
            return None
        return "<reminder>Refresh your current plan before continuing.</reminder>"

    def render(self) -> str:
        """
        把計劃渲染成可讀文本
        """
        if not self.state.items:
            return "No session plan yet."

        lines = []
        for item in self.state.items:
            marker = {
                "pending": "[ ]",
                "in_progress": "[>]",
                "completed": "[x]",
            }[item["status"]]
            line = f"{marker} {item['content']}"
            if item["status"] == "in_progress" and item["activeForm"]:
                line += f" ({item['activeForm']})"
            lines.append(line)

        completed = sum(1 for item in self.state.items if item["status"] == "completed")
        lines.append("")
        lines.append(f"({completed}/{len(self.state.items)} completed)")
        return "\n".join(lines)


In [ ]:
TODO = TodoManager()

## S02 Tools

> "加一个工具, 只加一个 handler" -- 循环不用动, 新工具注册进 dispatch map 就行。

只有 bash 时, 所有操作都走 shell。cat 截断不可预测, sed 遇到特殊字符就崩, 每次 bash 调用都是不受约束的安全面。专用工具 (read_file, write_file) 可以在工具层面做路径沙箱。

关键洞察: 加工具不需要改循环

**Process**

```text
+--------+      +-------+      +------------------+
|  User  | ---> |  LLM  | ---> | Tool Dispatch    |
| prompt |      |       |      | {                |
+--------+      +---+---+      |   bash: run_bash |
                    ^           |   read: run_read |
                    |           |   write: run_wr  |
                    +-----------+   edit: run_edit |
                    tool_result | }                |
                                +------------------+

The dispatch map is a dict: {tool_name: handler_function}.
One lookup replaces any if/elif chai
```

**Tools**

1. **Read**: 安全找到檔案 → 讀文字 → 拆成行 → 必要時只留前幾行 → 再限制總長度 → 回傳

In [ ]:
def safe_path(p: str) -> Path:
    """
    Get path of working directory safely
    """
    # 把使用者給的檔案路徑，限制在目前專案資料夾內。
    # 在 pathlib.Path 裡，/ 被重載成「組合路徑」，再由resolve把路徑轉成真正的絕對路徑
    path = (WORKDIR / p).resolve()
    # 檢查 path 是不是位於 WORKDIR 底下。
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

In [8]:
def run_bash(command: str) -> str:
    """
    Tool-1: run bash
    """
    # Risky commands
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command will be blocked"
    try:
        # 讓LLM執行外部指令
        # 標準輸出存進 r.stdout，錯誤輸出存進 r.stderr
        r = subprocess.run(command,
                           shell=True,
                           cwd=WORKDIR,
                           capture_output=True,
                           text=True,
                           timeout=120
                           )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout(120s)"

In [ ]:
def run_read(path: str, limit: int = None) -> str:
    """
    Tool 2: Read the file
    """
    try: 
        text = safe_path(path).read_text()
        lines = text.splitlines()
        # 安全地讀取某個檔案，必要時只回傳前幾行，避免太長。
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def run_write(path: str, content: str) -> str:
    """
    Tool 3: Write the file
    """
    try:
        fp = safe_path(path)
        # 如果父資料夾不存在，就幫你建立起來。(Parent表示如果中間好幾層資料夾不存在，也一起建立。)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Work {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"

In [11]:
def run_edit(path: str, old_text: str, new_text: str) -> str:
    """
    Tool 4: Edit the file
    """
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# -- The dispatch map: {tool_name: handler} --
# 如果模型說要呼叫某個工具名稱，那實際上要執行哪個 Python 函式。
TOOL_HANDLERS = {
    # kw 接收任意數量的 keyword arguments，並把它們包成一個 dict
    "bash": lambda **kw: run_bash(kw["command"]),
    "read_file": lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
    "todo": lambda **kw: TODO.update(kw["items"])
}

In [ ]:
# 提供給LLM的工具箱 (name + description + input_schema) = 工具規格書
TOOLS = [
    {
        "name": "bash",
        "description": "Run a shell command.",
        "input_schema": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
    {
        "name": "read_file",
        "description": "Read file contents.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "limit": {"type": "integer"},
            },
            "required": ["path"],
        },
    },
    {
        "name": "write_file",
        "description": "Write content to a file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
        },
    },
    {
        "name": "edit_file",
        "description": "Replace exact text in a file once.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "old_text": {"type": "string"},
                "new_text": {"type": "string"},
            },
            "required": ["path", "old_text", "new_text"],
        },
    },
    {
        "name": "todo",
        "description": "Rewrite the current session plan for multi-step work.",
        "input_schema": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "content": {"type": "string"},
                            "status": {
                                "type": "string",
                                "enum": ["pending", "in_progress", "completed"],
                            },
                            "activeForm": {
                                "type": "string",
                                "description": "Optional present-continuous label.",
                            },
                        },
                        "required": ["content", "status"],
                    },
                },
            },
            "required": ["items"],
        },
    },
]

In [ ]:
def extract_text(content) -> str:
    if not isinstance(content, list):
        return ""
    texts = []
    for block in content:
        text = getattr(block, "text", None)
        if text:
            texts.append(text)
    return "\n".join(texts).strip()

## S01 Agent Loop

> 一个退出条件控制整个流程。循环持续运行, 直到模型不再调用工具。

**Process**

```text
+--------+      +-------+      +---------+
|  User  | ---> |  LLM  | ---> |  Tool   |
| prompt |      |       |      | execute |
+--------+      +---+---+      +----+----+
                    ^                |
                    |   tool_result  |
                    +----------------+
                    (loop until stop_reason != "tool_use")
```

1. 先用 load_dotenv(override=True) 載入 .env
2. 用 os.environ / os.getenv() 取得 API 設定
3. 用 os.getcwd() 知道目前專案目錄
4. 使用者在終端機輸入問題 `message: {"role": "user"}`
5. 模型如果要用工具，就要求執行 bash `def run_bash`
6. subprocess.run(...) 在目前資料夾執行指令
7. 把指令結果再送回模型
8. 模型輸出最終文字答案 `message: {"role": "assistant"}`
9. 只把 text block 印出來 `print(block.text)`

In [ ]:
def agent_loop(messages):
    """
    The main agent loop with bash
    """
    while True:
        response = client.messages.create(
            model=MODEL,
            system=SYSTEM,
            messages=messages,
            tools=TOOLS,
            max_tokens=MAX_TOKEN
        )
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        if response.stop_reason != "tool_use":
            return response

        used_todo = False
        results = []
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "todo":
                    used_todo = True
                handler = TOOL_HANDLERS.get(block.name)
                output = handler(**block.input) if handler else f"Unknown tool: {block.name}"
                print(f"> {block.name}: {output[:200]}")
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": output,
                })

        if used_todo:
            TODO.mark_updated()
        else:
            TODO.note_round_without_update()

        content_blocks = []
        reminder = TODO.reminder()
        if reminder:
            content_blocks.append({
                "type": "text",
                "text": reminder,
            })
        content_blocks.extend(results)

        messages.append({
            "role": "user",
            "content": content_blocks
        })

In [14]:
if __name__ == "__main__":
    """
    Main Process
    """
    history = []
    while True:
        try:
            # 印出彩色提示字，然後等待你輸入內容
            query = input("\033[36ms01 >> \033[0m")
        # 提前終止程序 (Ctrl + C 或是 Ctrl + D終止)
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break
        history.append({
            "role": "user",
            "content": query
        })
        final_response = agent_loop(history)
        response_content = final_response.content if final_response else []
        if isinstance(response_content, list):
            for block in response_content:
                if getattr(block, "type", None) == "text" and hasattr(block, "text"):
                    print(block.text)
        print()

> write_file: Work 81 bytes to /Users/rong/Workspaces/2-Areas/21-Claude-Code/210-Harness-Agents/math.py
Done! Created `math.py` with an `add` function that takes two parameters and returns their sum.

> read_file: Error: [Errno 2] No such file or directory: '/Users/rong/Workspaces/2-Areas/21-Claude-Code/210-Harness-Agents/requirements.txt'
The `requirements.txt` file doesn't exist in that directory. Would you like me to create one?

> read_file: [project]
name = "210-harness-agents"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
    "pathlib>=1.0.1",
]

[dependency-
Here's the `pyproject.toml` file. It's a Python project called "210-harness-agents" with Python 3.12+ required and dependencies including anthropic, ipykernel, pathlib, and python-dotenv.

> write_file: Work 46 bytes to /Users/rong/Workspaces/2-Areas/21-Claude-Code/210-Harness-Agents/greet.py
> edit_file: Error: 'PosixPath' object has no attribute